# Sturm Manifest Builder v3.0
**Proyecto:** Inferencia del envejecimiento mediante modelo latente multimodal  
**Propósito:** Construir un manifest reproducible que una imágenes brightfield con el CSV central de biomarcadores (Lifespan_Study_Data.csv) y los metadatos de GEO (RNA-seq, metilación).  

### Cambios vs v2
- **4 parsers separados por fase** (PhaseI, II, III, V tienen formatos completamente distintos)
- **Join key corregido:** `(cell_line_norm, passage_int)` — el treatment es metadata, no parte del join
- **Tabla de normalización de treatments** para typos y variantes
- **PhaseV soportado:** formato `{fecha}_{microscopio}_{cell_line}_p{num}_{O2}Ox_{mag}x.jpeg`
- **Exclusiones explícitas:** Countess, Contamination, pre-study, other_treatments
- **Integración GEO metadata** desde los CSV de GSE179847 y GSE179848

## 0. CONFIG — Ajusta estas 5 rutas antes de correr

In [1]:
from pathlib import Path

# ─── AJUSTA ESTAS RUTAS ────────────────────────────────────────────────────────
CSV_PATH         = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv')
IMAGES_ROOT      = Path(r'/Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images')
GEO_RNA_META     = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/GSE179848_sample_metadata.csv')
GEO_METH_META    = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/GSE179847_sample_metadata.csv')
OUTPUT_DIR       = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS')
# ──────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config OK')
print(f'  CSV:        {CSV_PATH}')
print(f'  Imágenes:   {IMAGES_ROOT}')
print(f'  GEO RNA:    {GEO_RNA_META}')
print(f'  GEO Meth:   {GEO_METH_META}')
print(f'  Output:     {OUTPUT_DIR}')

Config OK
  CSV:        /Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv
  Imágenes:   /Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images
  GEO RNA:    /Users/JCB/Documentos/Proyecto Integrador/data/GSE179848_sample_metadata.csv
  GEO Meth:   /Users/JCB/Documentos/Proyecto Integrador/data/GSE179847_sample_metadata.csv
  Output:     /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS


## 1. Imports y utilidades

In [2]:
import re
import hashlib
import datetime
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupKFold

# ── Normalización de cell_line ────────────────────────────────────────────────
def normalize_cell_line(raw: str) -> str:
    """Normaliza a forma canónica: hFB1, hFB4, hFB5 ... hFB14.
    Elimina sufijos de género (-m, -f) para el join, pero los preserva
    como campo separado 'cell_line_sex'.
    """
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    # Extraer número y sufijo opcional
    m = re.match(r'(hFB(\d+))(?:-([mf]))?', s, re.IGNORECASE)
    if m:
        return f'hFB{int(m.group(2))}'
    return s.lower()

def extract_sex_suffix(raw: str) -> str:
    """Extrae sufijo de género si existe: 'm', 'f', o None."""
    if pd.isna(raw):
        return None
    m = re.search(r'hFB\d+-([mf])', str(raw), re.IGNORECASE)
    return m.group(1).lower() if m else None

# ── Normalización de treatments ───────────────────────────────────────────────
TREATMENT_MAP = {
    # Control variants (incluyendo typos)
    'ctrl': 'Control', 'control': 'Control', 'contorl': 'Control',
    # DEX
    'dex': 'DEX', 'acute dex': 'AcuteDEX',
    # MitoQ
    'mitoq': 'MitoQ', 'mitoq + dex': 'MitoQ+DEX', 'mitoq+dex': 'MitoQ+DEX',
    # NAC
    'nac': 'NAC', 'nac + dex': 'NAC+DEX', 'nac+dex': 'NAC+DEX',
    # Puls(atilla)
    'puls': 'Puls',
    # Alpha-ketoglutarate
    'a-keto': 'AKG', 'a-ketoglutarate': 'AKG',
    # Modulators (OxPhos)
    'mod': 'Modulators', 'modulators': 'Modulators',
    'mod+dex': 'Modulators+DEX', 'modulators+dex': 'Modulators+DEX',
    # Oligomycin
    'oligo': 'Oligomycin', 'oligomycin': 'Oligomycin',
    'oligo+dex': 'Oligomycin+DEX', 'oligomycin+dex': 'Oligomycin+DEX',
    # Otros
    '5-azacytidine': '5-Azacytidine',
    'contact inhibition': 'ContactInhibition',
    'contact inhibtion': 'ContactInhibition',  # typo en dataset
    'arrival': 'Arrival',
    'vehicle': 'Vehicle',
    'post-thaw': 'PostThaw',
}

def normalize_treatment(raw: str) -> str:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return 'Control'
    key = str(raw).strip().lower()
    return TREATMENT_MAP.get(key, str(raw).strip())

print('Utilidades cargadas OK')

Utilidades cargadas OK


## 2. Cargar y estandarizar el CSV central
El CSV tiene shape (1919, 271). Columnas clave identificadas en inspección previa.

In [70]:
# ════════════════════════════════════════════════════════════════════════════════
# SECCIÓN 2 — Cargar y estandarizar el CSV central
# Reemplaza la celda completa que empieza con:
#   "df_raw = pd.read_csv(CSV_PATH, ...)"
# ════════════════════════════════════════════════════════════════════════════════
 
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f'CSV shape: {df_raw.shape}')
 
# ── Mapa de columnas ──────────────────────────────────────────────────────────
# Usamos 'Treatments' (sin sufijo _21/_3) como treatment limpio — igual que Sturm
# 'Treatment' tiene el sufijo de O2 concatenado (Control_21, DEX_3, etc.)
# 'Percent_Oxygen' está separado como columna numérica
COL_MAP = {
    'Sample':                    'sample_id',
    'Cell_Line':                 'cell_line_raw',
    'Passage':                   'passage',
    'Study_Part':                'study_part',
    'Percent_Oxygen':            'percent_oxygen',
    'Treatments':                'treatment_clean',   # versión sin sufijo O2
    'Treatment':                 'treatment_full',    # versión con sufijo _21/_3
    'Population_Doublings_DI':   'pdl',
    'Percent_Lifespan':          'percent_lifespan',
    'Clinical_Condition':        'clinical_condition',
    'Donor_Age':                 'donor_age',
    'Cell_Type':                 'cell_type',
    'Days_Grown':                'days_grown',
    'Time_Point':                'time_point',        # para Contact Inhibition T0/T20/etc
    'Telomere_Length':           'telomere_length',
    'Copy_Number':               'mtdna_cn',
    'Cell_Size':                 'cell_size',
    'Cell_Volume':               'cell_volume',
    'Baseline_Respiration':      'seahorse_respiration',
    'Baseline_ECAR':             'seahorse_ecar',
    'RNAseq_ID':                 'rnaseq_id',
    'basename':                  'methylation_basename',
    # Relojes epigenéticos
    'Horvath1':                  'clock_horvath1',
    'PhenoAge':                  'clock_phenoage',
    'GrimAge':                   'clock_grimage',
    'Mitotic_Age':               'clock_mitotic',
    'DNAmTL':                    'clock_dnamtl',
    'DNAmSen':                   'clock_dnamsen',
    'DunedinPoAm_38':            'clock_dunedin',
}
 
existing_cols = {k: v for k, v in COL_MAP.items() if k in df_raw.columns}
missing_cols  = {k: v for k, v in COL_MAP.items() if k not in df_raw.columns}
if missing_cols:
    print(f'⚠️  Columnas no encontradas: {list(missing_cols.keys())}')
 
df = df_raw[list(existing_cols.keys())].rename(columns=existing_cols).copy()
 
# ── Normalización de tipos ────────────────────────────────────────────────────
df['cell_line']      = df['cell_line_raw'].apply(normalize_cell_line)
df['cell_line_sex']  = df['cell_line_raw'].apply(extract_sex_suffix)
df['passage']        = pd.to_numeric(df['passage'], errors='coerce')
df['pdl']            = pd.to_numeric(df.get('pdl',     pd.Series(np.nan, index=df.index)), errors='coerce')
df['study_part']     = pd.to_numeric(df.get('study_part', pd.Series(np.nan, index=df.index)), errors='coerce').astype('Int64')
df['percent_oxygen'] = pd.to_numeric(df.get('percent_oxygen', pd.Series(np.nan, index=df.index)), errors='coerce').astype('Int64')
df['time_point']     = pd.to_numeric(df.get('time_point', pd.Series(np.nan, index=df.index)), errors='coerce').astype('Int64')
 
# ── Normalizar treatment_clean para que coincida con parser de imágenes ───────
# El CSV usa 'Control', 'DEX', 'MitoQ+DEX', etc. en la columna 'Treatments'
# El parser de imágenes produce los mismos valores después de normalize_treatment
# → aplicar normalize_treatment al CSV también para garantizar consistencia
df['treatment_norm'] = df['treatment_clean'].apply(normalize_treatment)
 
# ── Validación de unicidad — DEBE PASAR antes de continuar ───────────────────
# La llave correcta es (cell_line, passage, treatment_norm, study_part, percent_oxygen)
# Contact Inhibition además necesita time_point
key_cols = ['cell_line', 'passage', 'treatment_norm', 'study_part', 'percent_oxygen']
dup_check = df.groupby(key_cols, dropna=False).size()
n_dups = (dup_check > 1).sum()
 
# Los únicos duplicados aceptables son Contact Inhibition (tienen time_point distinto)
if n_dups > 0:
    dup_examples = dup_check[dup_check > 1].reset_index()
    non_ci = dup_examples[~dup_examples['treatment_norm'].str.contains('Contact|Inhibit', na=False)]
    if len(non_ci) > 0:
        print(f'⛔ ERROR: {len(non_ci)} quíntuplas duplicadas fuera de Contact Inhibition:')
        print(non_ci.head(5))
    else:
        ci_dups = len(dup_examples)
        print(f'✅ Unicidad OK — {ci_dups} duplicados son Contact Inhibition (esperado, tienen time_point)')
else:
    print('✅ Unicidad perfecta — todas las quíntuplas son únicas')
 
print(f'\nDF estandarizado: {df.shape}')
print(f'sample_id únicos: {df["sample_id"].nunique()} / {len(df)}')
print(f'Cell lines: {sorted(df["cell_line"].dropna().unique())}')
print(f'Study_Parts: {sorted(df["study_part"].dropna().unique())}')
print(f'Percent_Oxygen: {sorted(df["percent_oxygen"].dropna().unique())}')
print(f'PDL range: {df["pdl"].min():.1f} – {df["pdl"].max():.1f}')
print(f'\nTreatments normalizados en CSV:')
print(sorted(df['treatment_norm'].dropna().unique()))
print(f'\nDisponibilidad de modalidades:')
for col, name in [('rnaseq_id','RNA-seq'), ('methylation_basename','Metilación'),
                  ('telomere_length','Telómero'), ('mtdna_cn','mtDNA CN')]:
    if col in df.columns:
        n = df[col].notna().sum()
        print(f'  {name:20s}: {n:4d} / {len(df)} ({100*n/len(df):.0f}%)')
 

CSV shape: (1919, 271)
⛔ ERROR: 1 quíntuplas duplicadas fuera de Contact Inhibition:
  cell_line  passage treatment_norm  study_part  percent_oxygen   0
7       NaN      NaN        Control           2              21  18

DF estandarizado: (1919, 32)
sample_id únicos: 1919 / 1919
Cell lines: ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2', 'hFB6', 'hFB7', 'hFB8', 'hek293']
Study_Parts: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Percent_Oxygen: [np.int64(3), np.int64(21)]
PDL range: 0.0 – 346.7

Treatments normalizados en CSV:
['2DG', '5-Azacytidine', '5-azacytidine+mitoNUITs', 'Contact_Inhibition', 'Contact_Inhibition_Regrowth', 'Control', 'DEX', 'Galactose', 'MitoQ', 'MitoQ+DEX', 'NAC', 'NAC+DEX', 'Oligomycin', 'Oligomycin+DEX', 'Pulsated_DEX', 'a-ketogluturate', 'betahydroxybutyrate', 'mitoNUITs', 'mitoNUITs+DEX']

Disponibilidad de modalidades:
  RNA-seq             :  345 / 1919 (18%)
  Metilación          :  496 / 1919 (26%)
  Telómero            :  496 

## 3. PDL normalizado por cell_line + bins early/mid/late

In [71]:
# PDL normalizado DENTRO de cada cell_line (0=más joven, 1=más viejo)
def normalize_pdl_per_line(group):
    mn, mx = group['pdl'].min(), group['pdl'].max()
    if mx - mn < 1e-6:
        return pd.Series(0.5, index=group.index)
    return (group['pdl'] - mn) / (mx - mn)

df['pdl_norm'] = df.groupby('cell_line', group_keys=False)['pdl'].transform(
    lambda g: (g - g.min()) / (g.max() - g.min() + 1e-9)
)

# Bins tercilos por cell_line: early / mid / late
def assign_bin(val, q33, q66):
    if pd.isna(val): return 'unknown'
    if val <= q33:   return 'early'
    if val <= q66:   return 'mid'
    return 'late'

pdl_bins = []
for cl, group in df.groupby('cell_line'):
    valid = group['pdl_norm'].dropna()
    q33 = valid.quantile(0.333) if len(valid) >= 3 else 0.333
    q66 = valid.quantile(0.666) if len(valid) >= 3 else 0.666
    bins = group['pdl_norm'].apply(lambda v: assign_bin(v, q33, q66))
    pdl_bins.append(bins)

df['pdl_bin'] = pd.concat(pdl_bins).reindex(df.index)

print('PDL normalizado OK')
print(df.groupby('cell_line')[['pdl','pdl_norm']].agg(['min','max','count']).round(2))
print('\nDistribución de bins:')
print(df.groupby(['cell_line','pdl_bin']).size().unstack(fill_value=0))

PDL normalizado OK
           pdl               pdl_norm           
           min     max count      min  max count
cell_line                                       
hFB1       0.0   60.36   218      0.0  1.0   218
hFB11      0.0   37.86    63      0.0  1.0    63
hFB12      0.0   68.45   496      0.0  1.0   496
hFB13      0.0   80.84   482      0.0  1.0   482
hFB14      0.0   44.94   213      0.0  1.0   213
hFB2       0.0   34.00   120      0.0  1.0   120
hFB6       0.0   25.23    68      0.0  1.0    68
hFB7       0.0   36.70    76      0.0  1.0    76
hFB8       0.0   34.30   109      0.0  1.0   109
hek293     0.0  346.67    55      0.0  1.0    55

Distribución de bins:
pdl_bin    early  late  mid  unknown
cell_line                           
hFB1          73    73   72        0
hFB11         21    21   21        0
hFB12        165   166  165        0
hFB13        161   161  160        1
hFB14         71    71   71        0
hFB2          40    40   40        0
hFB6          23    23   

## 4. Parser de imágenes — 4 parsers por fase

### Patrones identificados:
| Fase | Separador | Formato |
|------|-----------|---------|
| PhaseI  | espacios  | `hFB1-m P10  t175 N days growth 10x, Treatment` |
| PhaseII | comas     | `hFB12, Treatment, P6, 500kcells, t175, N-days growth, 10x` |
| PhaseIII| comas     | `hFB12, Treatment, P6, 500kcells, N-days growth, 21%, 10x` |
| PhaseV  | guiones   | `2021-02-24_ASM-01_hFB8_p35_3Ox_20x` |

In [34]:


# ── Carpetas a excluir ────────────────────────────────────────────────────────
EXCLUDE_PATTERNS = [
    'other_treatments', 'pre-study', 'Countess', 'Contamination',
    'Contact Inhibtion', 'Contact Inhibition',
]

def should_exclude(path: Path) -> bool:
    return any(excl in path.parts for excl in EXCLUDE_PATTERNS)

# ── REGEX por fase ────────────────────────────────────────────────────────────

# PhaseI estándar: "hFB1-m P10  t175 4 days growth 10x, Ctrl"
RE_P1 = re.compile(
    r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+[tT](\d+)\s+\d+\s+days?\s+growth\s+(\d+)x,?\s*(.*)$',
    re.IGNORECASE
)
# PhaseI flask al final: "hFB2-f P17 4 days growth T75 10x control"
RE_P1_T_END = re.compile(
    r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+\d+\s+days?\s+growth\s+[tT](\d+)\s+(\d+)x\s*(.*)$',
    re.IGNORECASE
)
# PhaseI sin flask: "hFB2-f P10 3 days growth 10x control"
RE_P1_SHORT = re.compile(
    r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+\d+\s+days?\s+growth\s+(\d+)x\s+(.+)$',
    re.IGNORECASE
)
# PhaseI orden invertido con days: "hFB1-m P4 10x 4 days growth T75"
RE_P1_MAG_FIRST = re.compile(
    r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+(\d+)x\s+\d+\s+days?\s+growth\s+[tT](\d+)$',
    re.IGNORECASE
)
# PhaseI orden invertido sin days: "hFB2-f P7 10x T175 4 days growth"
RE_P1_INV = re.compile(
    r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+(\d+)x\s+[tT](\d+)\s+\d+\s+days?\s+growth$',
    re.IGNORECASE
)
# PhaseI mínimo: "hFB1-m P5 10x T75" (sin days growth, sin treatment)
RE_P1_MINIMAL = re.compile(
    r'^(hFB\d+(?:-[mf])?)\s+P(\d+)\s+(\d+)x\s+[tT](\d+)$',
    re.IGNORECASE
)

# PhaseII con treatment y flask: "hFB12, ctrl, P6, 500kcells, t175, 5-days growth, 10x"
RE_P2_TREAT = re.compile(
    r'^(hFB\d+(?:-[mf])?),\s*([^,P][^,]*),\s*P(\d+),\s*([\d\.\-]+[kKmM]cells),\s*t(\d+),\s*[\d]+-days?\s+growth,?\s*(\d+)x$',
    re.IGNORECASE
)
# PhaseII sin treatment, con flask: "hFB10, P10, 1Mcells, t175, 3-days growth, 10x"
RE_P2_NOTREAT = re.compile(
    r'^(hFB\d+(?:-[mf])?),\s*P(\d+),\s*([\d\.\-]+[kKmM]cells),\s*t(\d+),\s*[\d]+-days?\s+growth,?\s*(\d+)x$',
    re.IGNORECASE
)
# PhaseII SIN flask (late passages): "hFB13, CTRL, P33, 1Mcells, 7-days growth, 20x"
# también cubre density con guion: "1-5Mcells"
RE_P2_NO_FLASK = re.compile(
    r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([\d\.\-]+[kKmM]cells),\s*[\d]+-days?\s+growth,\s*(\d+)x',
    re.IGNORECASE
)
# PhaseII con campos extra al final: "hFB14, CTRL, P31, 500kcells, 21-days growth, senescence, 20x"
# también: "hFB14, Oligo, P31, 600kcells, 21-days growth, senescence, 10x, fungi infection"
RE_P2_EXTRA = re.compile(
    r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([\d\.\-]+[kKmM]cells),\s*[\d]+-days?\s+growth,\s*[^,\d]+,\s*(\d+)x',
    re.IGNORECASE
)

# PhaseIII estándar con o2: "hFB13, DEX, P10, 500kcells, 7-days growth, 3%, 10x"
RE_P3 = re.compile(
    r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([^,]+cells),\s*[\d]+-days?\s+growth,\s*(\d+)%,\s*(\d+)x$',
    re.IGNORECASE
)
# PhaseIII con timepoint: "hFB12, Contact Inhibtion, P6, 100kcells, 8-days growth, 3%, T0, 10x"
RE_P3_TPOINT = re.compile(
    r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([^,]+cells),\s*[\d]+-days?\s+growth,\s*(\d+)%,\s*T\d+,\s*(\d+)x$',
    re.IGNORECASE
)
# PhaseIII sin o2_percent, con/sin post-thaw:
# "hFB11, ctrl, P7, 500kcells, 3-days growth, 10x, post-thaw"
RE_P3_NOO2 = re.compile(
    r'^(hFB\d+(?:-[mf])?),\s*([^,]+),\s*P(\d+),\s*([^,]+cells),\s*[\d]+-days?\s+growth,\s*(\d+)x(?:,.*)?$',
    re.IGNORECASE
)

# PhaseV estándar: "2021-02-24_ASM-01_hFB8_p35_3Ox_20x"
RE_P5 = re.compile(
    r'^(\d{4}-\d{2}-\d{2})_(ASM-\d+)_(hFB\d+(?:-[mf])?)_p(\d+)_(\d+)Ox_(\d+)x$',
    re.IGNORECASE
)
# PhaseV sufijo numérico: "2020-10-21_ASM-01_hFB12_p5_21Ox_10x-2"
RE_P5_DUP = re.compile(
    r'^(\d{4}-\d{2}-\d{2})_(ASM-\d+)_(hFB\d+(?:-[mf])?)_p(\d+)_(\d+)Ox_(\d+)x-\d+$',
    re.IGNORECASE
)
# PhaseV cualquier sufijo: "...20x_accumulation" o "...10x_4" o "...10x_1"
RE_P5_SUFFIX = re.compile(
    r'^(\d{4}-\d{2}-\d{2})_(ASM-\d+)_(hFB\d+(?:-[mf])?)_p(\d+)_(\d+)Ox_(\d+)x_.+$',
    re.IGNORECASE
)

# ── Función principal de parsing ──────────────────────────────────────────────
def parse_image_filename(stem: str, phase: str) -> dict | None:

    if phase == 'PhaseI':
        # Orden de intentos: de más específico a más general
        for pattern in [RE_P1, RE_P1_T_END, RE_P1_SHORT, RE_P1_INV, RE_P1_MAG_FIRST, RE_P1_MINIMAL]:
            m = pattern.match(stem)
            if not m:
                continue
            g = m.groups()
            if pattern == RE_P1:
                cl, pas, flask, mag, treat = g[0], g[1], g[2], g[3], g[4]
            elif pattern == RE_P1_T_END:
                cl, pas, flask, mag, treat = g[0], g[1], g[2], g[3], g[4] if g[4] else 'Control'
            elif pattern == RE_P1_SHORT:
                cl, pas, mag, treat = g[0], g[1], g[2], g[3]
                flask = None
            elif pattern == RE_P1_INV:
                cl, pas, mag, flask = g[0], g[1], g[2], g[3]
                treat = 'Control'
            elif pattern == RE_P1_MAG_FIRST:
                cl, pas, mag, flask = g[0], g[1], g[2], g[3]
                treat = 'Control'
            elif pattern == RE_P1_MINIMAL:
                cl, pas, mag, flask = g[0], g[1], g[2], g[3]
                treat = 'Control'
            return {
                'cell_line_raw':   cl,
                'passage':         int(pas),
                'flask_size':      int(flask) if flask else None,
                'magnification':   int(mag),
                'treatment_raw':   str(treat).strip(),
                'o2_percent':      None,
                'seeding_density': None,
                'microscope':      None,
            }
        return None

    elif phase == 'PhaseII':
        # Orden: con flask primero (más específico), sin flask después
        m = RE_P2_TREAT.match(stem)
        if m:
            return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                    'passage': int(m.group(3)), 'seeding_density': m.group(4),
                    'flask_size': int(m.group(5)), 'magnification': int(m.group(6)),
                    'o2_percent': None, 'microscope': None}
        m = RE_P2_NOTREAT.match(stem)
        if m:
            return {'cell_line_raw': m.group(1), 'treatment_raw': 'Control',
                    'passage': int(m.group(2)), 'seeding_density': m.group(3),
                    'flask_size': int(m.group(4)), 'magnification': int(m.group(5)),
                    'o2_percent': None, 'microscope': None}
        m = RE_P2_EXTRA.match(stem)
        if m:
            return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                    'passage': int(m.group(3)), 'seeding_density': m.group(4),
                    'flask_size': None, 'magnification': int(m.group(5)),
                    'o2_percent': None, 'microscope': None, 'parse_note': 'extra_fields'}
        m = RE_P2_NO_FLASK.match(stem)
        if m:
            return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                    'passage': int(m.group(3)), 'seeding_density': m.group(4),
                    'flask_size': None, 'magnification': int(m.group(5)),
                    'o2_percent': None, 'microscope': None}
        return None

    elif phase == 'PhaseIII':
        m = RE_P3.match(stem)
        if m:
            return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                    'passage': int(m.group(3)), 'seeding_density': m.group(4),
                    'flask_size': None, 'magnification': int(m.group(6)),
                    'o2_percent': int(m.group(5)), 'microscope': None}
        m = RE_P3_TPOINT.match(stem)
        if m:
            return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                    'passage': int(m.group(3)), 'seeding_density': m.group(4),
                    'flask_size': None, 'magnification': int(m.group(6)),
                    'o2_percent': int(m.group(5)), 'microscope': None}
        m = RE_P3_NOO2.match(stem)
        if m:
            return {'cell_line_raw': m.group(1), 'treatment_raw': m.group(2).strip(),
                    'passage': int(m.group(3)), 'seeding_density': m.group(4),
                    'flask_size': None, 'magnification': int(m.group(5)),
                    'o2_percent': None, 'microscope': None}
        return None

    elif phase == 'PhaseV':
        for pattern in [RE_P5, RE_P5_DUP, RE_P5_SUFFIX]:
            m = pattern.match(stem)
            if m:
                return {'cell_line_raw': m.group(3), 'treatment_raw': 'Control',
                        'passage': int(m.group(4)), 'flask_size': None,
                        'magnification': int(m.group(6)), 'o2_percent': int(m.group(5)),
                        'seeding_density': None, 'microscope': m.group(2),
                        'image_date': m.group(1)}
        return None

    return None

# ── Tests completos ───────────────────────────────────────────────────────────
tests = [
    # PhaseI
    ('PhaseI',   'hFB1-m P10  t175 4 days growth 10x, Ctrl'),
    ('PhaseI',   'hFB2-f P11  t175 5 days growth 20x, a-keto'),
    ('PhaseI',   'hFB2-f P10 3 days growth 10x control'),
    ('PhaseI',   'hFB2-f P17 4 days growth T75 10x control'),
    ('PhaseI',   'hFB2-f P7 10x T175 4 days growth'),
    ('PhaseI',   'hFB1-m P5 10x T75'),
    ('PhaseI',   'hFB1-m P4 10x 4 days growth T75'),
    # PhaseII
    ('PhaseII',  'hFB12, ctrl, P6, 500kcells, t175, 5-days growth, 10x'),
    ('PhaseII',  'hFB10, P10, 1Mcells, t175, 3-days growth, 10x'),
    ('PhaseII',  'hFB14, Modulators+DEX, P10, 500kcells, t175, 5-days growth, 20x'),
    ('PhaseII',  'hFB14, CTRL, P31, 500kcells, 21-days growth, senescence, 20x'),
    ('PhaseII',  'hFB14, Oligo, P31, 600kcells, 21-days growth, senescence, 10x, fungi infection'),
    ('PhaseII',  'hFB13, CTRL, P33, 1Mcells, 7-days growth, 20x'),
    ('PhaseII',  'hFB13, Mod+DEX, P33, 1-5Mcells, 7-days growth, 10x'),
    # PhaseIII
    ('PhaseIII', 'hFB13, DEX, P10, 500kcells, 7-days growth, 3%, 10x'),
    ('PhaseIII', 'hFB12, ctrl, P11, 1Mcells, 7-days growth, 3%, 10x'),
    ('PhaseIII', 'hFB12, Contact Inhibtion, P6, 100kcells, 8-days growth, 3%, T0, 10x'),
    ('PhaseIII', 'hFB11, ctrl, P7, 500kcells, 3-days growth, 10x, post-thaw'),
    # PhaseV
    ('PhaseV',   '2021-02-24_ASM-01_hFB8_p35_3Ox_20x'),
    ('PhaseV',   '2021-01-04_ASM-01_hFB12_p20_21Ox_10x'),
    ('PhaseV',   '2020-10-21_ASM-01_hFB12_p5_21Ox_10x-2'),
    ('PhaseV',   '2021-03-24_ASM-01_hFB12_p29_3Ox_20x_accumulation'),
    ('PhaseV',   '2020-11-18_ASM-01_hFB6_p17_3Ox_20x_4'),
]

print('── Tests de parsers v4 ──')
passed = 0
for phase, stem in tests:
    r = parse_image_filename(stem, phase)
    if r:
        cl  = normalize_cell_line(r['cell_line_raw'])
        tr  = normalize_treatment(r['treatment_raw'])
        p   = r['passage']
        mag = r['magnification']
        note = f'  [{r["parse_note"]}]' if r.get('parse_note') else ''
        print(f'  ✅ [{phase}] cell={cl}, pass={p}, treat={tr}, mag={mag}x{note}')
        passed += 1
    else:
        print(f'  ❌ [{phase}] FAIL: "{stem}"')

print(f'\n{passed}/{len(tests)} tests pasados')

── Tests de parsers v4 ──
  ✅ [PhaseI] cell=hFB1, pass=10, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB2, pass=11, treat=AKG, mag=20x
  ✅ [PhaseI] cell=hFB2, pass=10, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB2, pass=17, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB2, pass=7, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB1, pass=5, treat=Control, mag=10x
  ✅ [PhaseI] cell=hFB1, pass=4, treat=Control, mag=10x
  ✅ [PhaseII] cell=hFB12, pass=6, treat=Control, mag=10x
  ✅ [PhaseII] cell=hFB10, pass=10, treat=Control, mag=10x
  ✅ [PhaseII] cell=hFB14, pass=10, treat=Modulators+DEX, mag=20x
  ✅ [PhaseII] cell=hFB14, pass=31, treat=Control, mag=20x  [extra_fields]
  ✅ [PhaseII] cell=hFB14, pass=31, treat=Oligomycin, mag=10x  [extra_fields]
  ✅ [PhaseII] cell=hFB13, pass=33, treat=Control, mag=20x
  ✅ [PhaseII] cell=hFB13, pass=33, treat=Modulators+DEX, mag=10x
  ✅ [PhaseIII] cell=hFB13, pass=10, treat=DEX, mag=10x
  ✅ [PhaseIII] cell=hFB12, pass=11, treat=Control, mag=10x
  ✅ [PhaseIII] cell=h

## 5. Escaneo de imágenes

In [37]:
records = []
failed  = []
excluded = 0

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.tif', '.tiff', '.png'}

# Detectar la fase desde el path
def get_phase(path: Path) -> str | None:
    for part in path.parts:
        if re.match(r'Phase[IVX]+', part, re.IGNORECASE):
            # Normalizar: PhaseI, PhaseII, PhaseIII, PhaseV
            p = part.upper()
            if 'PHASEV' in p and 'PHASEVI' not in p:  # PhaseV pero no PhaseVI
                return 'PhaseV'
            if 'PHASEIII' in p: return 'PhaseIII'
            if 'PHASEII' in p:  return 'PhaseII'
            if 'PHASEI' in p:   return 'PhaseI'
    return None

print('Escaneando imágenes...')
for img_path in IMAGES_ROOT.rglob('*'):
    if img_path.suffix.lower() not in VALID_EXTENSIONS:
        continue
    if should_exclude(img_path):
        excluded += 1
        continue

    phase = get_phase(img_path)
    if phase is None:
        failed.append({'path': str(img_path), 'reason': 'no_phase_detected'})
        continue

    stem = img_path.stem
    parsed = parse_image_filename(stem, phase)

    if parsed is None:
        failed.append({'path': str(img_path), 'phase': phase, 'stem': stem, 'reason': 'parse_fail'})
        continue

    # Normalizar campos
    cell_line = normalize_cell_line(parsed['cell_line_raw'])
    sex       = extract_sex_suffix(parsed['cell_line_raw'])
    treatment = normalize_treatment(parsed['treatment_raw'])

    records.append({
        'img_path':        str(img_path),
        'phase':           phase,
        'cell_line':       cell_line,
        'cell_line_sex':   sex,
        'passage':         parsed['passage'],
        'treatment':       treatment,
        'treatment_raw':   parsed['treatment_raw'],
        'magnification':   parsed['magnification'],
        'flask_size':      parsed.get('flask_size'),
        'o2_percent':      parsed.get('o2_percent'),
        'seeding_density': parsed.get('seeding_density'),
        'microscope':      parsed.get('microscope'),
        'image_date':      parsed.get('image_date'),
        'extension':       img_path.suffix.lower(),
    })

df_imgs = pd.DataFrame(records)
df_failed = pd.DataFrame(failed)

print(f'\nImágenes parseadas:  {len(df_imgs):,}')
print(f'Imágenes excluidas:  {excluded:,}')
print(f'Imágenes fallidas:   {len(df_failed):,}')

if len(df_imgs) > 0:
    print('\nDistribución por fase:')
    print(df_imgs.groupby('phase').size())
    print('\nCell lines encontradas:')
    print(sorted(df_imgs['cell_line'].dropna().unique()))

Escaneando imágenes...

Imágenes parseadas:  2,398
Imágenes excluidas:  820
Imágenes fallidas:   49

Distribución por fase:
phase
PhaseI       425
PhaseII     1182
PhaseIII     168
PhaseV       623
dtype: int64

Cell lines encontradas:
['hFB1', 'hFB10', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2', 'hFB5', 'hFB6', 'hFB7', 'hFB8', 'hFB9']


## 5b. Diagnóstico de fallas del parser

In [38]:
if len(df_failed) > 0:
    print(f'=== {len(df_failed)} imágenes fallidas ===')
    print('\nDistribución de razones:')
    print(df_failed['reason'].value_counts())
    
    # Mostrar ejemplos agrupados por fase
    for phase in df_failed['phase'].dropna().unique() if 'phase' in df_failed.columns else []:
        subset = df_failed[df_failed['phase'] == phase]
        print(f'\n[{phase}] — {len(subset)} fallas. Primeros 5 stems:')
        for _, row in subset.head(5).iterrows():
            print(f'  "{row.get("stem", "")}"')
    
    # Sin fase detectada
    no_phase = df_failed[df_failed['reason'] == 'no_phase_detected']
    if len(no_phase) > 0:
        print(f'\nSin fase detectada ({len(no_phase)} files):')
        for _, row in no_phase.head(5).iterrows():
            print(f'  {row["path"]}')
else:
    print('✅ 0 fallas — todos los archivos parseados correctamente')

=== 49 imágenes fallidas ===

Distribución de razones:
reason
parse_fail    49
Name: count, dtype: int64

[PhaseV] — 6 fallas. Primeros 5 stems:
  "2020-11-18_ASM-01_hFB8_p19_21Ox_2.0x"
  "2021-03-31_ASM-01_hFB12_p29_3Ox_4x2_accumulations"
  "2021-03-31_ASM-01_hFB12_p29_3Ox_10x2_accumulations"
  "22021-04-07_ASM-01_hFB12_p30_21Ox_10x"
  "2021-04-07_ASM-01_hFB8_p38_3Ox_10x-acc2"

[PhaseII] — 43 fallas. Primeros 5 stems:
  "hFB8, ctrl, P15, 500kcells, t175, 7-days growth, 20x #2"
  "hFB12, Oligo, P9, 860kcells, t175, 5-days growth, 10x #2"
  "hFB12, Oligo, P9, 1Mcells, t175, 5-days growth, 20x #2"
  "hFB14, Oligo+DEX, P12, 1Mcells, t175, 6-days growth, 20x #2"
  "hFB12, Mod+DEX, P11, 1,5Mcells, t175, 6-days growth, 20x"


## 6. Join imágenes ↔ CSV central



In [73]:
# ════════════════════════════════════════════════════════════════════════════════
# SECCIÓN 6 — Join imágenes ↔ CSV central  (celda única, reemplaza todo)
# ════════════════════════════════════════════════════════════════════════════════

# ── Tabla de normalización EXTENDIDA ─────────────────────────────────────────
# Cubre todos los valores reales observados en df_imgs['treatment']
TREATMENT_MAP_EXTENDED = {
    # Control variants
    'ctrl': 'Control', 'control': 'Control', 'contorl': 'Control',
    '(1m), control': 'Control', '(2m), control': 'Control',
    '(700k) - control': 'Control', '': 'Control',

    # DEX variants
    'dex': 'DEX',
    '(1m), dex': 'DEX', '(2m), dex': 'DEX', '(2m),  dex': 'DEX',
    '(2m) - dex': 'DEX', 'dex vf': 'DEX',

    # Pulsated DEX
    'puls': 'Pulsated_DEX', 'dex-puls': 'Pulsated_DEX',
    '(1m), puls': 'Pulsated_DEX', '(2m), puls': 'Pulsated_DEX',
    '(2m),  puls': 'Pulsated_DEX', '(500k) - puls': 'Pulsated_DEX',
    '(500k), puls': 'Pulsated_DEX',

    # MitoQ
    'mitoq': 'MitoQ',
    '(1m), mitoq': 'MitoQ', '(1m) - mitoq': 'MitoQ', '(2m), mitoq': 'MitoQ',

    # MitoQ+DEX
    'mitoq+dex': 'MitoQ+DEX', 'mitoq + dex': 'MitoQ+DEX',
    '(1m), mitoq + dex': 'MitoQ+DEX', '(2m), mitoq + dex': 'MitoQ+DEX',
    '(2m) - mitoq + dex': 'MitoQ+DEX', '(2m), mq + dex': 'MitoQ+DEX',

    # NAC
    'nac': 'NAC',
    '(1m), nac': 'NAC', '(2m), nac': 'NAC', '(750k) - nac': 'NAC',

    # NAC+DEX
    'nac+dex': 'NAC+DEX', 'nac + dex': 'NAC+DEX', 'nac +dex': 'NAC+DEX',
    '(1m), nac + dex': 'NAC+DEX', '(2m), nac + dex': 'NAC+DEX',
    '(2m) - nac + dex': 'NAC+DEX',

    # AKG / a-ketoglutarate
    'akg': 'a-ketoglutarate', 'a-keto': 'a-ketoglutarate',
    'a-ketoglutarate': 'a-ketoglutarate',
    '(1m), a-keto': 'a-ketoglutarate', '(1m) - a-keto': 'a-ketoglutarate',
    '(2m), a-keto': 'a-ketoglutarate',

    # Oligomycin
    'oligomycin': 'Oligomycin', 'oligo': 'Oligomycin',

    # Oligomycin+DEX
    'oligomycin+dex': 'Oligomycin+DEX', 'oligo+dex': 'Oligomycin+DEX',

    # Modulators → Oligomycin (mismo compuesto en PhaseII)
    'modulators': 'Oligomycin', 'mod': 'Oligomycin',
    'modulators+dex': 'Oligomycin+DEX', 'mod+dex': 'Oligomycin+DEX',
    'mod-dex': 'Oligomycin+DEX',

    # Contact Inhibition
    'contactinhibition': 'Contact_Inhibition',
    'contact inhibition': 'Contact_Inhibition',
    'contact inhibtion': 'Contact_Inhibition',

    # Otros
    'posttthaw': 'Control', 'post-thaw': 'Control',
    'arrival': 'Control', 'vehicle': 'Control',
}

def normalize_treatment_v2(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return 'Control'
    key = str(raw).strip().lower()
    return TREATMENT_MAP_EXTENDED.get(key, str(raw).strip())

# ── Preparar df_imgs_join ────────────────────────────────────────────────────
PHASE_TO_STUDY_PART = {'PhaseI': 1, 'PhaseII': 2, 'PhaseIII': 3, 'PhaseV': 5}

df_imgs_join = df_imgs.copy()
df_imgs_join['passage']       = pd.to_numeric(df_imgs_join['passage'], errors='coerce').astype('Int64')
df_imgs_join['study_part']    = df_imgs_join['phase'].map(PHASE_TO_STUDY_PART).astype('Int64')
df_imgs_join['percent_oxygen']= df_imgs_join['o2_percent'].fillna(21).astype('Int64')
df_imgs_join['treatment_norm']= df_imgs_join['treatment'].apply(normalize_treatment_v2)
df_imgs_join['time_point']    = pd.NA  # imágenes no tienen time_point salvo CI

# Verificar cobertura de la normalización
unmapped = df_imgs_join[
    ~df_imgs_join['treatment'].isna() &
    (df_imgs_join['treatment_norm'] == df_imgs_join['treatment'])
]['treatment'].value_counts()
unmapped = unmapped[~unmapped.index.str.lower().isin(TREATMENT_MAP_EXTENDED.keys())]
if len(unmapped) > 0:
    print(f'⚠️  Treatments sin mapear ({len(unmapped)} valores únicos):')
    print(unmapped.head(15))
else:
    print('✅ Todos los treatments de imágenes mapeados')

# ── Preparar df_csv_join ─────────────────────────────────────────────────────
csv_keep_cols = ['sample_id', 'cell_line', 'passage', 'treatment_norm', 'study_part',
                 'percent_oxygen', 'time_point', 'pdl', 'pdl_norm', 'pdl_bin',
                 'percent_lifespan', 'clinical_condition', 'donor_age', 'cell_type',
                 'days_grown', 'telomere_length', 'mtdna_cn', 'cell_size', 'cell_volume',
                 'seahorse_respiration', 'seahorse_ecar', 'rnaseq_id', 'methylation_basename'] + \
                [c for c in df.columns if c.startswith('clock_')]
csv_keep_cols = [c for c in csv_keep_cols if c in df.columns]

df_csv_join = df[csv_keep_cols].copy()
df_csv_join['passage']        = pd.to_numeric(df_csv_join['passage'], errors='coerce').astype('Int64')
df_csv_join['study_part']     = df_csv_join['study_part'].astype('Int64')
df_csv_join['percent_oxygen'] = df_csv_join['percent_oxygen'].astype('Int64')
df_csv_join['time_point']     = df_csv_join['time_point'].astype('Int64')

# ── Limpiar NaN en llaves ANTES de separar CI ────────────────────────────────
n_before = len(df_csv_join)
df_csv_join = df_csv_join.dropna(subset=['cell_line', 'passage']).copy()
print(f'CSV: {n_before} → {len(df_csv_join)} ({n_before - len(df_csv_join)} eliminadas por NaN en llaves)')

n_before = len(df_imgs_join)
df_imgs_join = df_imgs_join.dropna(subset=['cell_line', 'passage']).copy()
print(f'Imgs: {n_before} → {len(df_imgs_join)} ({n_before - len(df_imgs_join)} eliminadas por NaN en llaves)')

# ── Separar Contact Inhibition ────────────────────────────────────────────────
is_ci_img = df_imgs_join['treatment_norm'].str.contains('Contact_Inhibition', na=False)
is_ci_csv = df_csv_join['treatment_norm'].str.contains('Contact_Inhibition', na=False)

df_imgs_standard = df_imgs_join[~is_ci_img].copy()
df_imgs_ci       = df_imgs_join[is_ci_img].copy()
df_csv_standard  = df_csv_join[~is_ci_csv].copy()
df_csv_ci        = df_csv_join[is_ci_csv].copy()

print(f'\nEstándar: {len(df_imgs_standard):,} imgs / {len(df_csv_standard):,} CSV rows')
print(f'CI:       {len(df_imgs_ci):,} imgs / {len(df_csv_ci):,} CSV rows')

# ── Verificar unicidad antes del merge ───────────────────────────────────────
JOIN_KEYS_BASE = ['cell_line', 'passage', 'treatment_norm', 'study_part', 'percent_oxygen']

dups_std = df_csv_standard.duplicated(subset=JOIN_KEYS_BASE, keep=False)
if dups_std.sum() > 0:
    print(f'\n⛔ {dups_std.sum()} duplicados en CSV estándar — mostrando ejemplos:')
    print(df_csv_standard[dups_std][JOIN_KEYS_BASE + ['sample_id']].head(10))
    raise ValueError('Resolver duplicados antes de continuar')
else:
    print(f'✅ CSV estándar: quíntupla única en todas las filas')

# ── JOIN estándar ─────────────────────────────────────────────────────────────
df_joined_standard = df_imgs_standard.merge(
    df_csv_standard,
    on=JOIN_KEYS_BASE,
    how='left',
    validate='many_to_one',
)

# ── JOIN Contact Inhibition ───────────────────────────────────────────────────
# Las imágenes CI no tienen time_point del parser → join sin time_point
# traer todas las mediciones CI del mismo pasaje y tomar la de T0 como representativa
df_csv_ci_t0 = df_csv_ci.sort_values('time_point').drop_duplicates(
    subset=JOIN_KEYS_BASE, keep='first'
).copy()

df_joined_ci = df_imgs_ci.merge(
    df_csv_ci_t0,
    on=JOIN_KEYS_BASE,
    how='left',
    validate='many_to_one',
)

# ── Unir y validar ────────────────────────────────────────────────────────────
df_manifest = pd.concat([df_joined_standard, df_joined_ci], ignore_index=True)

print('\n' + '═' * 55)
print('VALIDACIONES DE INTEGRIDAD')
print('═' * 55)

n_imgs_total = len(df_imgs_join)
n_manifest   = len(df_manifest)
check1 = n_imgs_total == n_manifest
print(f'{"✅" if check1 else "⛔"} Filas conservadas: {n_manifest:,} (esperado {n_imgs_total:,})')

check2 = df_manifest.duplicated(subset=['img_path']).sum() == 0
print(f'{"✅" if check2 else "⛔"} img_path único: {df_manifest["img_path"].nunique():,} únicos')

n_matched = df_manifest['pdl'].notna().sum()
pct = 100 * n_matched / n_manifest
check3 = pct >= 85
print(f'{"✅" if check3 else "⚠️ "} Match PDL: {n_matched:,} / {n_manifest:,} ({pct:.1f}%)')

VALID_CELL_LINES = {'hFB1','hFB2','hFB6','hFB7','hFB8','hFB11','hFB12','hFB13','hFB14'}
unexpected = set(df_manifest['cell_line'].dropna().unique()) - VALID_CELL_LINES
check4 = len(unexpected) == 0
print(f'{"✅" if check4 else "⚠️ "} Cell lines válidas: {unexpected if unexpected else "todas OK"}')

print('═' * 55)

# ── Flags de modalidades ──────────────────────────────────────────────────────
df_manifest['has_image']       = True
df_manifest['has_pdl']         = df_manifest['pdl'].notna()
df_manifest['has_rna']         = df_manifest['rnaseq_id'].notna()         if 'rnaseq_id'            in df_manifest.columns else False
df_manifest['has_methylation'] = df_manifest['methylation_basename'].notna() if 'methylation_basename' in df_manifest.columns else False
df_manifest['has_telomere']    = df_manifest['telomere_length'].notna()   if 'telomere_length'      in df_manifest.columns else False
df_manifest['has_mtdna']       = df_manifest['mtdna_cn'].notna()          if 'mtdna_cn'             in df_manifest.columns else False
df_manifest['n_modalities']    = df_manifest[['has_rna','has_methylation',
                                               'has_telomere','has_mtdna']].astype(int).sum(axis=1)

print('\nModalidades en imágenes con PDL:')
with_pdl = df_manifest[df_manifest['has_pdl']]
for col in ['has_rna','has_methylation','has_telomere','has_mtdna']:
    n = with_pdl[col].sum()
    print(f'  {col:22s}: {n:4,} / {len(with_pdl):,}')



✅ Todos los treatments de imágenes mapeados
CSV: 1919 → 1901 (18 eliminadas por NaN en llaves)
Imgs: 2398 → 2398 (0 eliminadas por NaN en llaves)

Estándar: 2,386 imgs / 1,815 CSV rows
CI:       12 imgs / 86 CSV rows
✅ CSV estándar: quíntupla única en todas las filas

═══════════════════════════════════════════════════════
VALIDACIONES DE INTEGRIDAD
═══════════════════════════════════════════════════════
✅ Filas conservadas: 2,398 (esperado 2,398)
✅ img_path único: 2,398 únicos
✅ Match PDL: 2,173 / 2,398 (90.6%)
⚠️  Cell lines válidas: {'hFB9', 'hFB5', 'hFB10'}
═══════════════════════════════════════════════════════

Modalidades en imágenes con PDL:
  has_rna               :  360 / 2,173
  has_methylation       :  540 / 2,173
  has_telomere          :  528 / 2,173
  has_mtdna             :  538 / 2,173


## 6b. Diagnóstico de imágenes sin match (sin PDL)

In [ ]:
# ── 6b Diagnóstico sin match ──────────────────────────────────────────────────
unmatched = df_manifest[~df_manifest['has_pdl']]
if len(unmatched) > 0:
    print(f'\n=== {len(unmatched)} imágenes sin match ===')
    print('Por (cell_line, passage, treatment_norm, study_part, percent_oxygen):')
    print(unmatched.groupby(
        ['cell_line','passage','treatment_norm','study_part','percent_oxygen'],
        dropna=False
    ).size().sort_values(ascending=False).head(15))

    print('\nVerificando causa (¿key no existe o treatment no matchea?):')
    for _, row in unmatched.drop_duplicates(
        subset=['cell_line','passage','study_part','percent_oxygen']
    ).head(8).iterrows():
        in_csv = df_csv_join[
            (df_csv_join['cell_line']      == row['cell_line'])      &
            (df_csv_join['passage']        == row['passage'])        &
            (df_csv_join['study_part']     == row['study_part'])     &
            (df_csv_join['percent_oxygen'] == row['percent_oxygen'])
        ]
        if len(in_csv) == 0:
            print(f'  ❌ No existe en CSV: {row["cell_line"]} P{row["passage"]} '
                  f'SP={row["study_part"]} O2={row["percent_oxygen"]}%')
        else:
            treats = sorted(in_csv['treatment_norm'].unique())
            print(f'  ⚠️  Treatment mismatch: {row["cell_line"]} P{row["passage"]} '
                  f'SP={row["study_part"]} | img={row["treatment_norm"]} | csv={treats}')
else:
    print('\n✅ Todas las imágenes tienen match')

## 7. Integrar metadata de GEO (RNA-seq y Metilación)

In [41]:
# ── GSE179848: RNA-seq metadata ───────────────────────────────────────────────
if GEO_RNA_META.exists():
    df_rna_meta = pd.read_csv(GEO_RNA_META)
    print(f'RNA-seq metadata: {df_rna_meta.shape}')
    print(f'  Columnas: {list(df_rna_meta.columns[:10])}')
    
    # Identificar la columna que funciona como join key con 'rnaseq_id' del CSV
    # Típicamente: 'geo_accession', 'title', 'sample_id', o similar
    print('\n  Primeras filas:')
    print(df_rna_meta.head(3))
else:
    print(f'⚠️  No se encontró: {GEO_RNA_META}')
    df_rna_meta = None

print()

# ── GSE179847: Methylation metadata ──────────────────────────────────────────
if GEO_METH_META.exists():
    df_meth_meta = pd.read_csv(GEO_METH_META)
    print(f'Metilación metadata: {df_meth_meta.shape}')
    print(f'  Columnas: {list(df_meth_meta.columns[:10])}')
    print('\n  Primeras filas:')
    print(df_meth_meta.head(3))
else:
    print(f'⚠️  No se encontró: {GEO_METH_META}')
    df_meth_meta = None

RNA-seq metadata: (345, 42)
  Columnas: ['geo_sample_id', 'title', 'source_name', 'organism', 'geo_accession', 'char_study_sample_id', 'char_cell_line_group', 'char_unique_variable_name', 'char_cell_line', 'char_inhouse_cell_line_name']

  Primeras filas:
  geo_sample_id                                        title  \
0    GSM5435591  SURF1_1_sp2_SURF1_Mutation_Control_ox21_p11   
1    GSM5435592  SURF1_1_sp2_SURF1_Mutation_Control_ox21_p14   
2    GSM5435593  SURF1_1_sp2_SURF1_Mutation_Control_ox21_p17   

                          source_name      organism geo_accession  \
0  Primary Human Fibroblast (forearm)  Homo sapiens     GSE179848   
1  Primary Human Fibroblast (forearm)  Homo sapiens     GSE179848   
2  Primary Human Fibroblast (forearm)  Homo sapiens     GSE179848   

   char_study_sample_id                     char_cell_line_group  \
0                   430  SURF1_1_sp2_SURF1_Mutation_Control_ox21   
1                   433  SURF1_1_sp2_SURF1_Mutation_Control_ox21   
2     

In [49]:
# Una vez que veas las columnas arriba, ajusta las join keys aquí:
# ─────────────────────────────────────────────────────────────────
RNA_JOIN_KEY_GEO  = 'char_study_sample_id'
METH_JOIN_KEY_GEO = 'char_study_sample_id'
# ─────────────────────────────────────────────────────────────────

def normalize_join_key(series: pd.Series) -> pd.Series:
    """Convierte llaves de join a texto limpio y consistente."""
    s = series.astype('string').str.strip()
    s = s.replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA, '<NA>': pd.NA})
    # Si una llave numérica llega como 12345.0, convertirla a 12345
    s = s.str.replace(r'^(-?\d+)\.0$', r'\1', regex=True)
    return s

# Evitar duplicados si esta celda se ejecuta múltiples veces en la misma sesión
cols_to_drop = [c for c in df_manifest.columns if c.startswith('rna_') or c.startswith('meth_')]
if cols_to_drop:
    df_manifest = df_manifest.drop(columns=cols_to_drop)

if df_rna_meta is not None and RNA_JOIN_KEY_GEO in df_rna_meta.columns and 'rnaseq_id' in df_manifest.columns:
    # Prefixar columnas GEO para evitar colisiones
    rna_cols = {c: f'rna_{c}' for c in df_rna_meta.columns if c != RNA_JOIN_KEY_GEO}
    df_rna_join = df_rna_meta.rename(columns=rna_cols).copy()

    # Normalizar llaves de join en ambos lados
    df_manifest = df_manifest.copy()
    df_manifest['_rna_join_key'] = normalize_join_key(df_manifest['rnaseq_id'])
    df_rna_join['_rna_join_key'] = normalize_join_key(df_rna_join[RNA_JOIN_KEY_GEO])

    df_manifest = df_manifest.merge(
        df_rna_join.drop(columns=[RNA_JOIN_KEY_GEO]),
        on='_rna_join_key',
        how='left'
    ).drop(columns=['_rna_join_key'])

    print(f'RNA-seq metadata integrada: {df_manifest["rnaseq_id"].notna().sum()} filas con RNA_ID')
else:
    print('⚠️ RNA join omitido: revisa df_rna_meta / RNA_JOIN_KEY_GEO / rnaseq_id')

if df_meth_meta is not None and METH_JOIN_KEY_GEO in df_meth_meta.columns and 'methylation_basename' in df_manifest.columns:
    meth_cols = {c: f'meth_{c}' for c in df_meth_meta.columns if c != METH_JOIN_KEY_GEO}
    df_meth_join = df_meth_meta.rename(columns=meth_cols).copy()

    # Normalizar llaves de join en ambos lados
    df_manifest = df_manifest.copy()
    df_manifest['_meth_join_key'] = normalize_join_key(df_manifest['methylation_basename'])
    df_meth_join['_meth_join_key'] = normalize_join_key(df_meth_join[METH_JOIN_KEY_GEO])

    df_manifest = df_manifest.merge(
        df_meth_join.drop(columns=[METH_JOIN_KEY_GEO]),
        on='_meth_join_key',
        how='left'
    ).drop(columns=['_meth_join_key'])

    print(f'Metilación metadata integrada: {df_manifest["methylation_basename"].notna().sum()} filas con basename')
else:
    print('⚠️ Methylation join omitido: revisa df_meth_meta / METH_JOIN_KEY_GEO / methylation_basename')

print(f'\nManifest final: {df_manifest.shape}')

RNA-seq metadata integrada: 5333 filas con RNA_ID
Metilación metadata integrada: 7240 filas con basename

Manifest final: (22556, 117)


## 8. GroupKFold splits por cell_line (anti-leakage)

In [50]:
# Solo asignar splits a filas con PDL (las útiles para entrenar)
df_with_pdl = df_manifest[df_manifest['has_pdl']].copy()

# GroupKFold garantiza que un mismo cell_line no aparezca en train y val
# Esto es CRÍTICO — con solo ~9 donantes, el leakage por donante destruye la evaluación
gkf = GroupKFold(n_splits=5)
groups = df_with_pdl['cell_line'].values

df_with_pdl['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(gkf.split(df_with_pdl, groups=groups)):
    df_with_pdl.iloc[val_idx, df_with_pdl.columns.get_loc('fold')] = fold_idx

# Merge fold back al manifest completo
df_manifest = df_manifest.merge(
    df_with_pdl[['img_path','fold']],
    on='img_path', how='left'
)

print('Distribución de splits (GroupKFold por cell_line):')
print(df_with_pdl.groupby('fold')['cell_line'].unique().apply(lambda x: ', '.join(sorted(x))))
print()
print(df_with_pdl.groupby('fold').agg(
    n_images=('img_path','count'),
    n_cell_lines=('cell_line','nunique'),
    pdl_min=('pdl','min'),
    pdl_max=('pdl','max')
).round(1))

Distribución de splits (GroupKFold por cell_line):
fold
0                      hFB12
1                      hFB13
2                      hFB14
3                 hFB1, hFB6
4    hFB11, hFB2, hFB7, hFB8
Name: cell_line, dtype: str

      n_images  n_cell_lines  pdl_min  pdl_max
fold                                          
0         7721             1      2.2     67.2
1         6636             1      2.0     80.8
2         2800             1      0.0     44.9
3         2687             2      1.9     60.4
4         2684             4      1.5     36.7


## 9. Export manifest versionado con MD5

In [51]:
import hashlib, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Manifest completo
out_full = OUTPUT_DIR / f'manifest_full_{timestamp}.csv'
df_manifest.to_csv(out_full, index=False)

# Calcular MD5
md5 = hashlib.md5(open(out_full,'rb').read()).hexdigest()
print(f'Manifest completo guardado:')
print(f'  {out_full}')
print(f'  Shape: {df_manifest.shape}')
print(f'  MD5:   {md5}')

# Guardar reporte de metadatos
meta = {
    'timestamp':        timestamp,
    'n_total_images':   len(df_manifest),
    'n_with_pdl':       int(df_manifest['has_pdl'].sum()),
    'n_failed_parse':   len(df_failed),
    'n_excluded':       excluded,
    'cell_lines':       sorted(df_manifest['cell_line'].dropna().unique().tolist()),
    'phases':           sorted(df_manifest['phase'].dropna().unique().tolist()),
    'pdl_range':        [float(df_manifest['pdl'].min()), float(df_manifest['pdl'].max())],
    'md5':              md5,
    'csv_path':         str(CSV_PATH),
    'images_root':      str(IMAGES_ROOT),
}
import json
meta_path = OUTPUT_DIR / f'manifest_meta_{timestamp}.json'
json.dump(meta, open(meta_path,'w'), indent=2)
print(f'\nMetadata guardada: {meta_path}')

Manifest completo guardado:
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_full_20260315_163839.csv
  Shape: (297048, 118)
  MD5:   89c29477ea89341738414f7e9bb7796a

Metadata guardada: /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_meta_20260315_163839.json


## 10. Subconjunto MVP-1
Para el primer encoder visual: solo imágenes con PDL disponible, solo tratamiento Control, 10x.

Excluyendo SURF1 del baseline (donantes con enfermedad mitocondrial) — se pueden agregar luego para robustez.

In [52]:
mvp1 = df_manifest[
    df_manifest['has_pdl'] &
    (df_manifest['treatment'] == 'Control') &
    (df_manifest['magnification'] == 10)
].copy()

# Opcional: excluir SURF1 del baseline limpio
if 'clinical_condition' in mvp1.columns:
    n_before = len(mvp1)
    mvp1_clean = mvp1[mvp1['clinical_condition'].str.lower().str.contains('healthy|control|normal', na=True)]
    print(f'MVP-1 con SURF1:    {n_before:,}')
    print(f'MVP-1 solo healthy: {len(mvp1_clean):,}')
    print('(Guarda mvp1 para incluir SURF1 en experimentos posteriores)')
else:
    mvp1_clean = mvp1

print(f'\nMVP-1 baseline: {len(mvp1_clean):,} imágenes')
print(f'Cell lines:     {sorted(mvp1_clean["cell_line"].dropna().unique())}')
print(f'Fases:          {sorted(mvp1_clean["phase"].dropna().unique())}')
print(f'PDL range:      {mvp1_clean["pdl"].min():.1f} – {mvp1_clean["pdl"].max():.1f}')
print(f'PDL bins:')
print(mvp1_clean['pdl_bin'].value_counts())

# Guardar MVP-1
out_mvp1 = OUTPUT_DIR / f'manifest_mvp1_ctrl10x_{timestamp}.csv'
mvp1_clean.to_csv(out_mvp1, index=False)
md5_mvp1 = hashlib.md5(open(out_mvp1,'rb').read()).hexdigest()
print(f'\nMVP-1 guardado: {out_mvp1}')
print(f'MD5: {md5_mvp1}')

MVP-1 con SURF1:    61,840
MVP-1 solo healthy: 60,005
(Guarda mvp1 para incluir SURF1 en experimentos posteriores)

MVP-1 baseline: 60,005 imágenes
Cell lines:     ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2']
Fases:          ['PhaseI', 'PhaseII', 'PhaseIII', 'PhaseV']
PDL range:      0.0 – 80.8
PDL bins:
pdl_bin
early    26439
mid      20458
late     13108
Name: count, dtype: int64

MVP-1 guardado: /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_mvp1_ctrl10x_20260315_163839.csv
MD5: 606c6cb2332c4465304b527b3150e50b


## 11. Resumen final y checklist de calidad

In [53]:
print('='*60)
print('RESUMEN DEL MANIFEST')
print('='*60)
print(f'Imágenes totales parseadas: {len(df_manifest):,}')
print(f'Imágenes con PDL:           {df_manifest["has_pdl"].sum():,}')
print(f'Fallas de parser:           {len(df_failed):,}')
print(f'Imágenes excluidas (CI/etc): {excluded:,}')
print()
print('Por fase:')
print(df_manifest.groupby('phase').agg(
    n=('img_path','count'),
    con_pdl=('has_pdl','sum')
))
print()
print('CHECKLIST MVP-1')
print('-'*40)
checks = [
    ('Manifest con >1000 imágenes',    len(mvp1_clean) > 1000),
    ('PDL disponible en >80%',         df_manifest['has_pdl'].mean() > 0.8),
    ('Fallas parser <5%',              len(df_failed)/max(len(df_manifest),1) < 0.05),
    ('≥2 cell lines en MVP-1',         mvp1_clean['cell_line'].nunique() >= 2),
    ('Splits GroupKFold asignados',    'fold' in df_manifest.columns),
    ('early/mid/late bins presentes',  mvp1_clean['pdl_bin'].nunique() == 3),
]
for desc, passed in checks:
    icon = '✅' if passed else '❌'
    print(f'  {icon} {desc}')
print()
print('Archivos generados:')
print(f'  {out_full}')
print(f'  {out_mvp1}')
print(f'  {meta_path}')

RESUMEN DEL MANIFEST
Imágenes totales parseadas: 297,048
Imágenes con PDL:           296,540
Fallas de parser:           49
Imágenes excluidas (CI/etc): 820

Por fase:
               n  con_pdl
phase                    
PhaseI     30053    30053
PhaseII   145123   145117
PhaseIII   57984    57612
PhaseV     63888    63758

CHECKLIST MVP-1
----------------------------------------
  ✅ Manifest con >1000 imágenes
  ✅ PDL disponible en >80%
  ✅ Fallas parser <5%
  ✅ ≥2 cell lines en MVP-1
  ✅ Splits GroupKFold asignados
  ✅ early/mid/late bins presentes

Archivos generados:
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_full_20260315_163839.csv
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_mvp1_ctrl10x_20260315_163839.csv
  /Users/JCB/Documentos/Proyecto Integrador/data/MANIFESTS/manifest_meta_20260315_163839.json
